# TeachAgent 四部分总览

这份 notebook 用一个稳定的 demo 主链，把 TeachAgent 当前的四个部分串起来看一遍：

1. 知识树与叶子卡片
2. 错题绑定与学生确认
3. 遗忘复习系统
4. 学生长期记忆层

设计原则：

- 默认不依赖真实模型调用
- 默认不修改正式知识库
- 只往 `scratch/teachagent_system_overview/` 写 demo 产物
- 重点看主链是否打通，而不是单点极限效果


In [1]:
import copy
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from annotation_to_review_state import convert_annotations_to_review_state
from coach_agent import (
    CoachSession,
    ErrorType,
    analyze_student_reply,
    apply_student_memory_bias,
    build_reply_analysis_fallback,
    build_student_memory_context,
    build_student_memory_strategy_note,
    build_turn_prompt,
    get_coach_strategy,
    infer_completed_from_reply,
)
from review_bundle_builder import build_review_bundles, load_example_map, load_leaf_card_lookup
from review_scheduler import build_review_schedule
from review_state_manager import apply_review_action
from scripts.convert_leaf_draft_to_cards import load_inventory
from student_memory_events import (
    binding_result_to_memory_event,
    coach_response_to_memory_event,
    diagnosis_result_to_memory_event,
    review_state_update_to_memory_event,
    student_choice_to_memory_event,
)
from student_memory_store import append_event, build_profile_from_store, load_events, summarize_store
from wrong_question_binder import WrongQuestionBinder, summarize_result

STUDENT_ID = 'student_overview_demo'
SESSION_ID = 'session_overview_demo_001'
BASE_TIME = datetime(2026, 6, 24, 20, 0, tzinfo=timezone(timedelta(hours=8)))
DEMO_NOW = datetime(2026, 6, 24, 21, 0, tzinfo=timezone(timedelta(hours=8)))

SCRATCH_DIR = PROJECT_ROOT / 'scratch' / 'teachagent_system_overview'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

BINDER_RESULTS_JSON = SCRATCH_DIR / 'binder_results_demo.json'
ANNOTATION_JSON = SCRATCH_DIR / 'annotation_records_demo.json'
REVIEW_STATE_JSON = SCRATCH_DIR / 'review_state_demo.json'
LEAF_FIRST_BUNDLE_JSON = SCRATCH_DIR / 'review_bundles_leaf_first_demo.json'
QUESTION_FIRST_BUNDLE_JSON = SCRATCH_DIR / 'review_bundles_question_first_demo.json'
REVIEW_AFTER_NODE_JSON = SCRATCH_DIR / 'review_after_node_feedback_demo.json'
REVIEW_AFTER_QUESTION_JSON = SCRATCH_DIR / 'review_after_question_feedback_demo.json'
MEMORY_EVENTS_JSONL = SCRATCH_DIR / 'student_memory_events_demo.jsonl'
MEMORY_PROFILE_JSON = SCRATCH_DIR / 'student_memory_profile_demo.json'

EXAMPLES_MD_PATH = PROJECT_ROOT / 'docs' / 'rag_samples' / 'taizhou_simulated_exam_examples_batch_01.md'

def show_json(payload):
    print(json.dumps(payload, ensure_ascii=False, indent=2))

def show_block(title: str, text: str) -> None:
    print(f'\n{title}')
    print('-' * len(title))
    print(text.strip() if str(text).strip() else '（空）')

def write_json(path: Path, payload) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

def reset_demo_file(path: Path) -> None:
    if path.exists():
        path.unlink()

def find_primary_card(node_id: str, leaf_card_lookup: dict[str, list[dict]]) -> dict | None:
    rows = leaf_card_lookup.get(node_id, [])
    if not rows:
        return None
    for row in rows:
        if row.get('is_primary'):
            return row
    return rows[0]

def top_node_view(schedule, limit: int = 5) -> list[dict]:
    return [
        {
            'node_id': item['node_id'],
            'priority_score': item['priority_score'],
            'memory_priority_boost': item.get('memory_priority_boost'),
            'memory_priority_reason': item.get('memory_priority_reason'),
            'session_priority_boost': item.get('session_priority_boost'),
            'priority_note': item.get('priority_note'),
        }
        for item in schedule.node_priorities[:limit]
    ]

def top_question_view(schedule, limit: int = 5) -> list[dict]:
    return [
        {
            'question_id': item['question_id'],
            'priority_score': item['priority_score'],
            'last_result': item.get('last_result'),
            'memory_priority_boost': item.get('memory_priority_boost'),
            'memory_priority_reason': item.get('memory_priority_reason'),
            'session_priority_boost': item.get('session_priority_boost'),
            'priority_note': item.get('priority_note'),
        }
        for item in schedule.question_priorities[:limit]
    ]


## Part 1. 知识树与叶子卡片

这里先不重新生成卡片，而是直接读取当前已经建好的 inventory 和 leaf cards，确认这套“知识树 -> 叶子卡片”主层已经有可用产物。

In [2]:
INVENTORY = load_inventory()
LEAF_CARD_LOOKUP = load_leaf_card_lookup()

SAMPLE_NODE_IDS = [
    'math.数列与不等式.数列.递推数列转化方法.构造等比数列',
    'math.三角函数_平面向量与解三角形.平面向量.向量夹角公式',
    'math.解析几何.直线的方程.点到直线的距离公式',
]

sample_leaf_cards = []
for node_id in SAMPLE_NODE_IDS:
    meta = INVENTORY[node_id]
    card = find_primary_card(node_id, LEAF_CARD_LOOKUP)
    sample_leaf_cards.append(
        {
            'node_id': node_id,
            'name': meta.get('name'),
            'node_kind': meta.get('node_kind'),
            'review_role': meta.get('review_role'),
            'binding_role': meta.get('binding_role'),
            'path': meta.get('path'),
            'primary_card': {
                'chunk_id': card.get('chunk_id') if card else None,
                'card_type': card.get('card_type') if card else None,
                'title': card.get('title') if card else None,
                'review_cue': card.get('review_cue') if card else None,
                'text_preview': (card.get('text') or '')[:180] if card else None,
            },
        }
    )

show_json(
    {
        'inventory_leaf_count': sum(1 for item in INVENTORY.values() if item.get('is_leaf')),
        'leaf_card_node_count': len(LEAF_CARD_LOOKUP),
        'sample_leaf_cards': sample_leaf_cards,
    }
)


{
  "inventory_leaf_count": 279,
  "leaf_card_node_count": 279,
  "sample_leaf_cards": [
    {
      "node_id": "math.数列与不等式.数列.递推数列转化方法.构造等比数列",
      "name": "构造等比数列",
      "node_kind": "method",
      "review_role": "core",
      "binding_role": "primary_allowed",
      "path": [
        "数学",
        "数列与不等式",
        "数列",
        "递推数列转化方法",
        "构造等比数列"
      ],
      "primary_card": {
        "chunk_id": "chunk.math.数列与不等式.数列.递推数列转化方法.构造等比数列.method_card.v1",
        "card_type": "method_card",
        "title": "构造等比数列",
        "review_cue": "构造等比时，重点看能不能把递推式整理成“后一项是前一项乘一个固定因子”。",
        "text_preview": "方法：构造等比数列。路径：数学 > 数列与不等式 > 数列 > 递推数列转化方法 > 构造等比数列。别名：构造比值不变。关键词：构造等比；递推转化；常比；辅助数列。目标：通过乘除、移项或代换把原递推式转化成某个辅助数列满足固定公比，从而求解原数列。触发信号：递推式经过整理后可能出现“新量前后项比值恒定”；原数列不直接等比但有乘法型递推结构；题目适合构造辅助数"
      }
    },
    {
      "node_id": "math.三角函数_平面向量与解三角形.平面向量.向量夹角公式",
      "name": "向量夹角公式",
      "node_kind": "formula",
      "review_role": "core",
      "binding_role": "primary_allow

## Part 2. 错题绑定与学生确认

这一段分两步：

1. 系统先用 `wrong_question_binder.py` 给出推荐
2. 学生再沿知识树做最终确认

为了稳定复现，这里用现成 harness 里的三道题：数列、向量、解析几何。

In [3]:
FIXTURE_DIR = PROJECT_ROOT / 'harness' / 'fixtures' / 'wrong_binder'
CASE_FILES = [
    FIXTURE_DIR / 'binder_case_sequence_shift.json',
    FIXTURE_DIR / 'binder_case_vector_angle.json',
    FIXTURE_DIR / 'binder_case_circle_distance.json',
]
CASE_PAYLOADS = [json.loads(path.read_text(encoding='utf-8')) for path in CASE_FILES]

QUESTION_ID_MAP = {
    'binder_case_sequence_shift': 'wq_overview_seq_001',
    'binder_case_vector_angle': 'wq_overview_vec_001',
    'binder_case_circle_distance': 'wq_overview_circle_001',
}

MANUAL_SELECTIONS = {
    'binder_case_sequence_shift': {
        'primary_node_ids': ['math.数列与不等式.数列.递推数列转化方法.构造等比数列'],
        'secondary_node_ids': ['math.数列与不等式.数列.基础概念.数列递推公式解读'],
        'selection_notes': '学生确认这题主错点是递推转等比，递推公式解读作为辅助点。',
    },
    'binder_case_vector_angle': {
        'primary_node_ids': ['math.三角函数_平面向量与解三角形.平面向量.向量夹角公式'],
        'secondary_node_ids': ['math.三角函数_平面向量与解三角形.平面向量.数量积几何意义'],
        'selection_notes': '学生确认核心在夹角公式，数量积几何意义作为前置背景。',
    },
    'binder_case_circle_distance': {
        'primary_node_ids': ['math.解析几何.直线的方程.点到直线的距离公式'],
        'secondary_node_ids': ['math.解析几何.圆的方程.圆的标准方程'],
        'selection_notes': '学生确认这题最终落脚在点到直线距离，但必须联动圆的标准方程。',
    },
}

BINDER = WrongQuestionBinder(enable_embeddings=False)
BINDER_OBJECTS = []
BINDER_SUMMARY = []

for case in CASE_PAYLOADS:
    result = BINDER.bind({'question_payload': case['question_payload']})
    summary = summarize_result(result)
    BINDER_OBJECTS.append({'case': case, 'result': result, 'summary': summary})
    BINDER_SUMMARY.append(
        {
            'case_id': case['case_id'],
            'description': case['description'],
            'question_id': QUESTION_ID_MAP[case['case_id']],
            'primary_node_id': summary['primary_node_id'],
            'secondary_node_ids': summary['secondary_node_ids'],
            'binding_confidence': summary['binding_confidence'],
            'coarse_subtrees': summary['coarse_subtrees'],
            'top_k_preview': [
                {
                    'rank': item['rank'],
                    'node_id': item['node_id'],
                    'title': item['title'],
                    'bind_score': item['bind_score'],
                    'reason': item['reason'],
                }
                for item in result.binding_record['top_k_candidates'][:5]
            ],
        }
    )

write_json(BINDER_RESULTS_JSON, {'results': BINDER_SUMMARY})
show_json(BINDER_SUMMARY)


[
  {
    "case_id": "binder_case_sequence_shift",
    "description": "递推构造等比数列证明题",
    "question_id": "wq_overview_seq_001",
    "primary_node_id": "math.数列与不等式.数列.等差_等比数列.等比数列通项公式",
    "secondary_node_ids": [],
    "binding_confidence": 0.2688,
    "coarse_subtrees": [
      "math.数列与不等式.数列",
      "math.数列与不等式.数列.等差_等比数列",
      "math.数列与不等式.数列.基础概念"
    ],
    "top_k_preview": [
      {
        "rank": 1,
        "node_id": "math.数列与不等式.数列.等差_等比数列.等比数列通项公式",
        "title": "等比数列通项公式",
        "bind_score": 0.3476,
        "reason": "命中关键词：等比数列、公比、a_n；路径语境贴近：数列；解析模式匹配：等比数列；该叶子对题目核心求解动作的解释最完整"
      },
      {
        "rank": 2,
        "node_id": "math.数列与不等式.数列.等差_等比数列.等比数列性质与判定",
        "title": "等比数列性质与判定",
        "bind_score": 0.3311,
        "reason": "命中关键词：等比数列、公比；路径语境贴近：数列；解析模式匹配：等比数列"
      },
      {
        "rank": 3,
        "node_id": "math.数列与不等式.数列.等差_等比数列.等比数列求和公式",
        "title": "等比数列求和公式",
        "bind_score": 0.3228,
        "reason": "命中关键词：等比数列；路径语境贴近：

In [4]:
ANNOTATION_RECORDS = []

for index, item in enumerate(BINDER_OBJECTS):
    case = item['case']
    result = item['result']
    summary = item['summary']
    case_id = case['case_id']
    manual = MANUAL_SELECTIONS[case_id]
    primary_node_ids = manual['primary_node_ids']
    secondary_node_ids = manual['secondary_node_ids']
    selected_node_ids = primary_node_ids + secondary_node_ids

    annotation = {
        'annotation_id': f'anno_overview_{index + 1:03d}',
        'question_id': QUESTION_ID_MAP[case_id],
        'question_payload': case['question_payload'],
        'recommendation': {
            'mode': 'heuristic',
            'coarse_candidate_node_ids': summary['coarse_subtrees'],
            'candidate_leaf_node_ids': summary['top_k_node_ids'],
            'candidate_items': [
                {
                    'node_id': row['node_id'],
                    'title': row['title'],
                    'path': row['path'],
                    'rank': row['rank'],
                    'reason': row['reason'],
                }
                for row in result.binding_record['top_k_candidates'][:5]
            ],
            'notes': [
                f"系统主绑定：{summary['primary_node_id']}",
                '这一步只是推荐，学生确认才是最终绑定。',
            ],
        },
        'student_selection': {
            'selection_status': 'confirmed',
            'selected_node_ids': selected_node_ids,
            'primary_node_ids': primary_node_ids,
            'secondary_node_ids': secondary_node_ids,
            'mistake_node_ids': primary_node_ids,
            'selection_notes': manual['selection_notes'],
            'used_recommendation': True,
        },
        'review_linkage': {
            'review_node_ids': selected_node_ids,
            'review_notes': '确认后的知识点直接进入复习系统。',
        },
        'created_at': (BASE_TIME + timedelta(minutes=index * 15)).isoformat(),
    }
    ANNOTATION_RECORDS.append(annotation)

write_json(ANNOTATION_JSON, ANNOTATION_RECORDS)
show_json(ANNOTATION_RECORDS[0])


{
  "annotation_id": "anno_overview_001",
  "question_id": "wq_overview_seq_001",
  "question_payload": {
    "stem": "已知数列 a_n 满足 a_{n+1}=3a_n+1，且 a_1=1/2，求证：数列 {a_n+1/2} 为等比数列。",
    "question_type": "证明题",
    "student_answer": "我知道要从递推式入手，但不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列。",
    "correct_answer": "数列 {a_n+1/2} 是以 1 为首项、3 为公比的等比数列。",
    "solution_text": "由 a_{n+1}=3a_n+1，得 a_{n+1}+1/2=3a_n+1+1/2=3(a_n+1/2)。又 a_1+1/2=1/2+1/2=1，因此新数列满足后一项等于前一项乘 3，且首项为 1，所以 {a_n+1/2} 是首项为 1、公比为 3 的等比数列。",
    "teacher_comment": "核心卡点是把递推式通过同时加 1/2 变形成 b_{n+1}=3b_n，再用等比数列定义完成证明。",
    "source_name": "binder_harness",
    "source_section": "数列递推转等比",
    "tags": [
      "数列",
      "递推",
      "构造新数列",
      "等比数列"
    ]
  },
  "recommendation": {
    "mode": "heuristic",
    "coarse_candidate_node_ids": [
      "math.数列与不等式.数列",
      "math.数列与不等式.数列.等差_等比数列",
      "math.数列与不等式.数列.基础概念"
    ],
    "candidate_leaf_node_ids": [
      "math.数列与不等式.数列.等差_等比数列.等比数列通项公式",
      "math.数列与不等式.数列.等差_等比数列.等比数

## Part 3. 遗忘复习系统

这一步把学生确认后的题目记录转成 `review_state`，然后分别看：

- 先知识点后题目
- 先题目后知识点

再模拟学生按钮反馈，观察下一轮排序怎么变化。

In [5]:
REVIEW_STATE = convert_annotations_to_review_state(
    ANNOTATION_RECORDS,
    existing_review_state=None,
    record_id='review_state.teachagent_system_overview.v1',
    student_id=STUDENT_ID,
    source_batch_id='teachagent_system_overview_batch_01',
)
write_json(REVIEW_STATE_JSON, REVIEW_STATE)

EXAMPLE_MAP = load_example_map(EXAMPLES_MD_PATH)
LEAF_FIRST_BUNDLES = build_review_bundles(
    REVIEW_STATE,
    example_map=EXAMPLE_MAP,
    leaf_card_lookup=LEAF_CARD_LOOKUP,
    now=DEMO_NOW,
    mode='leaf_first',
    bundle_limit=5,
    bundle_question_limit=3,
)
QUESTION_FIRST_BUNDLES = build_review_bundles(
    REVIEW_STATE,
    example_map=EXAMPLE_MAP,
    leaf_card_lookup=LEAF_CARD_LOOKUP,
    now=DEMO_NOW,
    mode='question_first',
    bundle_limit=5,
    bundle_question_limit=3,
)

write_json(LEAF_FIRST_BUNDLE_JSON, LEAF_FIRST_BUNDLES.as_dict())
write_json(QUESTION_FIRST_BUNDLE_JSON, QUESTION_FIRST_BUNDLES.as_dict())

show_json(
    {
        'review_state_record_id': REVIEW_STATE.get('record_id'),
        'knowledge_point_count': len(REVIEW_STATE.get('knowledge_point_states', [])),
        'question_count': len(REVIEW_STATE.get('example_question_states', [])),
        'leaf_first_first_bundle': LEAF_FIRST_BUNDLES.review_bundles[0],
        'question_first_first_bundle': QUESTION_FIRST_BUNDLES.review_bundles[0],
    }
)


{
  "review_state_record_id": "review_state.teachagent_system_overview.v1",
  "knowledge_point_count": 6,
  "question_count": 3,
  "leaf_first_first_bundle": {
    "bundle_id": "bundle_01",
    "mode": "leaf_first",
    "rank": 1,
    "node_id": "math.三角函数_平面向量与解三角形.平面向量.向量夹角公式",
    "priority_score": 0.8975,
    "bundle_reason": "知识点已到复习时间，并联动 1 道练习题一起复习",
    "leaf_card": {
      "chunk_id": "chunk.math.三角函数_平面向量与解三角形.平面向量.向量夹角公式.formula_card.v1",
      "node_id": "math.三角函数_平面向量与解三角形.平面向量.向量夹角公式",
      "title": "向量夹角公式",
      "card_type": "formula_card",
      "node_kind": "formula",
      "path": [
        "数学",
        "三角函数、平面向量与解三角形",
        "平面向量",
        "向量夹角公式"
      ],
      "keywords": [
        "夹角公式",
        "cosθ",
        "数量积",
        "模长"
      ],
      "text": "知识点：向量夹角公式。路径：数学 > 三角函数、平面向量与解三角形 > 平面向量 > 向量夹角公式。别名：向量夹角。关键词：夹角公式；cosθ；数量积；模长。公式：对非零向量 a、b，有 cosθ=(a·b)/(|a||b|)。适用条件：两个向量都非零；需要求向量夹角或借夹角公式反求数量积、模长关系；已知坐标或几何量可求出 a·b、|a|、|b|。特殊情况：若 a·b=0，则夹角为 90°；求几何中的

In [6]:
SEQ_PRIMARY_NODE_ID = MANUAL_SELECTIONS['binder_case_sequence_shift']['primary_node_ids'][0]
CIRCLE_QUESTION_ID = QUESTION_ID_MAP['binder_case_circle_distance']

BASE_SCHEDULE = build_review_schedule(REVIEW_STATE, now=DEMO_NOW, user_mode='mixed')

NODE_UPDATE = apply_review_action(
    copy.deepcopy(REVIEW_STATE),
    action='node_needs_more_practice',
    target_type='knowledge_point',
    target_id=SEQ_PRIMARY_NODE_ID,
    now=DEMO_NOW,
)
NODE_SCHEDULE = build_review_schedule(NODE_UPDATE.updated_payload, now=DEMO_NOW, user_mode='mixed')
write_json(REVIEW_AFTER_NODE_JSON, NODE_UPDATE.as_dict())

QUESTION_UPDATE = apply_review_action(
    copy.deepcopy(REVIEW_STATE),
    action='review_result',
    target_type='wrong_question',
    target_id=CIRCLE_QUESTION_ID,
    result='wrong',
    now=DEMO_NOW + timedelta(hours=1),
)
QUESTION_SCHEDULE = build_review_schedule(QUESTION_UPDATE.updated_payload, now=DEMO_NOW + timedelta(hours=1), user_mode='mixed')
write_json(REVIEW_AFTER_QUESTION_JSON, QUESTION_UPDATE.as_dict())

show_json(
    {
        'before_feedback_top_nodes': top_node_view(BASE_SCHEDULE, limit=3),
        'after_node_needs_more_practice_top_nodes': top_node_view(NODE_SCHEDULE, limit=3),
        'before_feedback_top_questions': top_question_view(BASE_SCHEDULE, limit=3),
        'after_question_wrong_top_questions': top_question_view(QUESTION_SCHEDULE, limit=3),
    }
)


{
  "before_feedback_top_nodes": [
    {
      "node_id": "math.三角函数_平面向量与解三角形.平面向量.向量夹角公式",
      "priority_score": 0.8975,
      "memory_priority_boost": 0.0,
      "memory_priority_reason": null,
      "session_priority_boost": 0.0,
      "priority_note": "学生确认核心在夹角公式，数量积几何意义作为前置背景。"
    },
    {
      "node_id": "math.三角函数_平面向量与解三角形.平面向量.数量积几何意义",
      "priority_score": 0.8975,
      "memory_priority_boost": 0.0,
      "memory_priority_reason": null,
      "session_priority_boost": 0.0,
      "priority_note": "学生确认核心在夹角公式，数量积几何意义作为前置背景。"
    },
    {
      "node_id": "math.数列与不等式.数列.基础概念.数列递推公式解读",
      "priority_score": 0.8975,
      "memory_priority_boost": 0.0,
      "memory_priority_reason": null,
      "session_priority_boost": 0.0,
      "priority_note": "学生确认这题主错点是递推转等比，递推公式解读作为辅助点。"
    }
  ],
  "after_node_needs_more_practice_top_nodes": [
    {
      "node_id": "math.数列与不等式.数列.递推数列转化方法.构造等比数列",
      "priority_score": 1.085,
      "memory_priority_boost": 0.0,
      "me

## Part 4. 学生长期记忆层

最后把前面这条链真正沉淀成长期事件：

- 绑定事件
- 学生确认事件
- 诊断事件
- 引导事件
- 复习事件

然后从事件库重建 `student_memory_profile`，再看它如何轻量影响：

- `coach_agent` 的策略话术
- `review_scheduler` 的排序偏置


In [7]:
DIAGNOSIS_BY_CASE = {
    'binder_case_sequence_shift': {
        'error_type': 'missing_strategy',
        'confidence': 0.84,
        'reason': '学生知道要从递推式入手，但不会自己提出构造新数列这个中间目标。',
        'evidence': '我知道要从递推式入手，但不知道为什么要构造 a_n+1/2。',
        'coach_mode': 'socratic_light',
        'coach_trap': '学生知道局部知识，但不会先确定中间目标。',
        'coach_prompt': '先让学生明确中间目标，再推进具体变形。',
        'source': 'overview_demo_manual',
    },
    'binder_case_vector_angle': {
        'error_type': 'missing_strategy',
        'confidence': 0.79,
        'reason': '学生知道数量积，但不会先把向量和关系转成模平方结构。',
        'evidence': '我能想到数量积，但不会把向量和的模平方展开到夹角公式。',
        'coach_mode': 'socratic_light',
        'coach_trap': '学生会局部公式，但不会先做结构转换。',
        'coach_prompt': '先提醒把向量和关系改写成模平方，再问下一步。',
        'source': 'overview_demo_manual',
    },
    'binder_case_circle_distance': {
        'error_type': 'concept_gap',
        'confidence': 0.82,
        'reason': '学生不会把张角存在性先转成圆心到直线距离范围，几何判定概念不稳。',
        'evidence': '我知道要用点到直线距离，但不知道为什么先转成圆心到直线距离不超过 2。',
        'coach_mode': 'direct_explain',
        'coach_trap': '学生缺少必要概念，继续追问会放大挫败。',
        'coach_prompt': '先讲清存在性转化，再回到距离公式。',
        'source': 'overview_demo_manual',
    },
}

COACH_BY_CASE = {
    'binder_case_sequence_shift': {
        'content': '我们先别急着算结果，先想这道题最想把递推式改成什么前后项关系。你先说说，新数列最好变成哪种结构？',
        'reply_quality': 'empty',
        'strategy': {
            'mode': 'socratic_light',
            'trap': '学生知道局部知识，但不会先确定中间目标。',
            'prompt': '先让学生明确中间目标，再推进具体变形。',
        },
        'turn_index': 1,
        'done': False,
        'stop_reason': 'continue',
        'reply_analysis': {
            'quality': 'empty',
            'understands': False,
            'completed': False,
            'reason': '学生还说不出“后一项等于前一项乘固定因子”这个中间目标。',
        },
    },
    'binder_case_vector_angle': {
        'content': '这题不是直接套夹角公式，而是先把 a+b=-c 变成长度关系。你先试着说说，为什么这里先考虑 |a+b|^2 会更方便？',
        'reply_quality': 'weak',
        'strategy': {
            'mode': 'socratic_light',
            'trap': '学生会局部公式，但不会先做结构转换。',
            'prompt': '先点出结构转换，再只追问一个关键中间量。',
        },
        'turn_index': 1,
        'done': False,
        'stop_reason': 'continue',
        'reply_analysis': {
            'quality': 'weak',
            'understands': True,
            'completed': False,
            'reason': '学生已经想到数量积，但还不会主动把向量和改写成模平方。',
        },
    },
    'binder_case_circle_distance': {
        'content': '这题关键不是先算 t，而是先判断什么情况下直线上能找到这样的点 P。你先记住一个核心转化：张角存在性先变成圆心到直线的距离范围。',
        'reply_quality': 'good',
        'strategy': {
            'mode': 'direct_explain',
            'trap': '学生缺少必要概念，继续追问会放大挫败。',
            'prompt': '先讲清存在性转化，再回到距离公式。',
        },
        'turn_index': 1,
        'done': False,
        'stop_reason': 'continue',
        'reply_analysis': {
            'quality': 'good',
            'understands': True,
            'completed': False,
            'reason': '学生已经接住了“先转成距离范围”这个核心概念，但还没把后续不等式写完。',
        },
    },
}

REVIEW_RESULT_BY_CASE = {
    'binder_case_sequence_shift': 'wrong',
    'binder_case_vector_angle': 'partial',
    'binder_case_circle_distance': 'wrong',
}

reset_demo_file(MEMORY_EVENTS_JSONL)
MEMORY_REVIEW_STATE = copy.deepcopy(REVIEW_STATE)
WRITTEN_EVENTS = []

for index, record in enumerate(ANNOTATION_RECORDS):
    case_id = CASE_PAYLOADS[index]['case_id']
    question = record['question_payload']
    selection = record['student_selection']
    question_id = record['question_id']
    primary_node_id = selection['primary_node_ids'][0]
    secondary_node_ids = selection['secondary_node_ids']
    selected_node_ids = selection['selected_node_ids']
    occurred_base = BASE_TIME + timedelta(minutes=index * 25)

    binding_event = binding_result_to_memory_event(
        question_id=question_id,
        question_type=question.get('question_type'),
        primary_node_id=primary_node_id,
        secondary_node_ids=secondary_node_ids,
        candidate_node_ids=selected_node_ids,
        binding_source='student_confirmed',
        occurred_at=occurred_base.isoformat(),
        source_name=question.get('source_name'),
        source_section=question.get('source_section'),
        student_id=STUDENT_ID,
        session_id=SESSION_ID,
        binding_confidence=0.9,
    )
    append_event(binding_event, path=MEMORY_EVENTS_JSONL)
    WRITTEN_EVENTS.append(binding_event)

    choice_event = student_choice_to_memory_event(
        action_type='select_node',
        target_type='question',
        target_id=question_id,
        occurred_at=(occurred_base + timedelta(minutes=1)).isoformat(),
        question_id=question_id,
        question_type=question.get('question_type'),
        selected_node_ids=selected_node_ids,
        note=selection.get('selection_notes'),
        source_name=question.get('source_name'),
        source_section=question.get('source_section'),
        student_id=STUDENT_ID,
        session_id=SESSION_ID,
    )
    append_event(choice_event, path=MEMORY_EVENTS_JSONL)
    WRITTEN_EVENTS.append(choice_event)

    diagnosis_event = diagnosis_result_to_memory_event(
        DIAGNOSIS_BY_CASE[case_id],
        question_id=question_id,
        question_type=question.get('question_type'),
        binding={
            'primary_node_id': primary_node_id,
            'secondary_node_ids': secondary_node_ids,
            'linked_node_ids': selected_node_ids,
        },
        occurred_at=(occurred_base + timedelta(minutes=3)).isoformat(),
        source_name=question.get('source_name'),
        source_section=question.get('source_section'),
        student_id=STUDENT_ID,
        session_id=SESSION_ID,
    )
    append_event(diagnosis_event, path=MEMORY_EVENTS_JSONL)
    WRITTEN_EVENTS.append(diagnosis_event)

    coach_event = coach_response_to_memory_event(
        COACH_BY_CASE[case_id],
        question_id=question_id,
        question_type=question.get('question_type'),
        binding={
            'primary_node_id': primary_node_id,
            'secondary_node_ids': secondary_node_ids,
            'linked_node_ids': selected_node_ids,
        },
        occurred_at=(occurred_base + timedelta(minutes=6)).isoformat(),
        error_type=DIAGNOSIS_BY_CASE[case_id]['error_type'],
        source_name=question.get('source_name'),
        source_section=question.get('source_section'),
        student_id=STUDENT_ID,
        session_id=SESSION_ID,
    )
    append_event(coach_event, path=MEMORY_EVENTS_JSONL)
    WRITTEN_EVENTS.append(coach_event)

    review_update = apply_review_action(
        MEMORY_REVIEW_STATE,
        action='review_result',
        target_type='wrong_question',
        target_id=question_id,
        result=REVIEW_RESULT_BY_CASE[case_id],
        now=occurred_base + timedelta(days=2),
    )
    MEMORY_REVIEW_STATE = review_update.updated_payload
    review_event = review_state_update_to_memory_event(
        review_update.as_dict(),
        student_id=STUDENT_ID,
        session_id=SESSION_ID,
    )
    append_event(review_event, path=MEMORY_EVENTS_JSONL)
    WRITTEN_EVENTS.append(review_event)

practice_update = apply_review_action(
    MEMORY_REVIEW_STATE,
    action='node_needs_more_practice',
    target_type='knowledge_point',
    target_id=SEQ_PRIMARY_NODE_ID,
    now=BASE_TIME + timedelta(days=3),
)
MEMORY_REVIEW_STATE = practice_update.updated_payload
practice_event = review_state_update_to_memory_event(
    practice_update.as_dict(),
    student_id=STUDENT_ID,
    session_id=SESSION_ID,
)
append_event(practice_event, path=MEMORY_EVENTS_JSONL)
WRITTEN_EVENTS.append(practice_event)

PROFILE = build_profile_from_store(student_id=STUDENT_ID, path=MEMORY_EVENTS_JSONL)
write_json(MEMORY_PROFILE_JSON, PROFILE)

show_json(
    {
        'store_summary': summarize_store(MEMORY_EVENTS_JSONL).as_dict(),
        'event_count': len(load_events(path=MEMORY_EVENTS_JSONL, student_id=STUDENT_ID)),
        'personalization_summary': PROFILE.get('personalization_summary'),
        'question_memories_preview': PROFILE.get('question_memories', [])[:3],
        'node_memories_preview': PROFILE.get('node_memories', [])[:3],
    }
)


{
  "store_summary": {
    "path": "/Users/xumuchi/Desktop/TeachAgent/scratch/teachagent_system_overview/student_memory_events_demo.jsonl",
    "total_events": 16,
    "event_type_counts": {
      "binding": 3,
      "coach": 3,
      "diagnosis": 3,
      "review": 4,
      "student_choice": 3
    },
    "distinct_students": 1
  },
  "event_count": 16,
  "personalization_summary": {
    "dominant_error_type": "missing_strategy",
    "dominant_error_signal_strength": "tentative",
    "dominant_error_share": 0.6667,
    "memory_stage": "forming_pattern",
    "observation_counts": {
      "diagnosis_event_count": 3,
      "coach_event_count": 3,
      "review_event_count": 4,
      "distinct_question_count": 3
    },
    "top_error_types": [
      {
        "error_type": "missing_strategy",
        "count": 2,
        "signal_strength": "tentative"
      },
      {
        "error_type": "concept_gap",
        "count": 1,
        "signal_strength": "tentative"
      }
    ],
    "top_recu

In [8]:
TARGET_RECORD = ANNOTATION_RECORDS[0]
TARGET_QUESTION = TARGET_RECORD['question_payload']
TARGET_REPLY = TARGET_QUESTION['student_answer']
TARGET_ERROR_TYPE = ErrorType.MISSING_STRATEGY
TARGET_STUDENT_PROFILE = '学生会从递推式入手，但经常不会自己提出构造辅助数列这种中间目标。'

reply_analysis = build_reply_analysis_fallback(
    analyze_student_reply(TARGET_REPLY),
    '本总览 notebook 为了稳定对比，只使用本地 fallback，不调用 reply analyzer 模型。',
    completed=infer_completed_from_reply(TARGET_REPLY),
)

base_strategy = get_coach_strategy(
    TARGET_ERROR_TYPE,
    reply_analysis.quality,
    turn_index=0,
    total_turns=4,
    understands=reply_analysis.understands,
    completed=reply_analysis.completed,
)

session_plain = CoachSession(
    problem_text=TARGET_QUESTION['stem'],
    error_type=TARGET_ERROR_TYPE,
    student_profile=TARGET_STUDENT_PROFILE,
    initial_student_reply=TARGET_REPLY,
)
session_memory = CoachSession(
    problem_text=TARGET_QUESTION['stem'],
    error_type=TARGET_ERROR_TYPE,
    student_profile=TARGET_STUDENT_PROFILE,
    student_memory_profile=PROFILE,
    student_memory_context=build_student_memory_context(PROFILE),
    initial_student_reply=TARGET_REPLY,
)

strategy_plain = apply_student_memory_bias(base_strategy, session_plain)
strategy_memory = apply_student_memory_bias(base_strategy, session_memory)

prompt_plain = build_turn_prompt(
    session=session_plain,
    student_reply=TARGET_REPLY,
    reply_analysis=reply_analysis,
    strategy=strategy_plain,
)
prompt_memory = build_turn_prompt(
    session=session_memory,
    student_reply=TARGET_REPLY,
    reply_analysis=reply_analysis,
    strategy=strategy_memory,
)

schedule_plain = build_review_schedule(
    MEMORY_REVIEW_STATE,
    now=BASE_TIME + timedelta(days=3, hours=1),
    user_mode='mixed',
    student_memory_profile=None,
)
schedule_memory = build_review_schedule(
    MEMORY_REVIEW_STATE,
    now=BASE_TIME + timedelta(days=3, hours=1),
    user_mode='mixed',
    student_memory_profile=PROFILE,
)

show_json(
    {
        'coach_compare': {
            'reply_analysis': reply_analysis.as_dict(),
            'strategy_without_memory': strategy_plain.as_dict(),
            'strategy_with_memory': strategy_memory.as_dict(),
            'memory_note_added': build_student_memory_strategy_note(base_strategy, session_memory),
            'prompt_without_memory_preview': prompt_plain[:280],
            'prompt_with_memory_preview': prompt_memory[:320],
        },
        'review_compare': {
            'top_nodes_without_memory': top_node_view(schedule_plain, limit=5),
            'top_nodes_with_memory': top_node_view(schedule_memory, limit=5),
            'top_questions_without_memory': top_question_view(schedule_plain, limit=5),
            'top_questions_with_memory': top_question_view(schedule_memory, limit=5),
        },
    }
)

show_block('有长期画像时的 student_memory_context', session_memory.student_memory_context)
show_block('无长期画像：本轮策略话术', strategy_plain.as_prompt_block())
show_block('有长期画像：本轮策略话术', strategy_memory.as_prompt_block())
show_block('长期画像额外加上的策略提示', build_student_memory_strategy_note(base_strategy, session_memory))


{
  "coach_compare": {
    "reply_analysis": {
      "quality": "empty",
      "understands": false,
      "completed": false,
      "reason": "本总览 notebook 为了稳定对比，只使用本地 fallback，不调用 reply analyzer 模型。",
      "source": "fallback_heuristic"
    },
    "strategy_without_memory": {
      "mode": "socratic_light",
      "trap": "学生知道局部知识，但不会先确定中间目标。",
      "prompt": "学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。"
    },
    "strategy_with_memory": {
      "mode": "socratic_light",
      "trap": "学生知道局部知识，但不会先确定中间目标。",
      "prompt": "学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。结合长期画像：优先先讲清概念、定义或判定依据，再回到当前题。"
    },
    "memory_note_added": "结合长期画像：优先先讲清概念、定义或判定依据，再回到当前题。",
    "prompt_without_memory_preview": "【题目】\n已知数列 a_n 满足 a_{n+1}=3a_n+1，且 a_1=1/2，求证：数列 {a_n+1/2} 为等比数列。\n\n【学生画像】\n学生会从递推式入手，但经常不会自己提出构造辅助数列这种中间目标。\n\n【学生错误类型】\nmissing_strategy\n\n【学生最新回答】\n我知道要从递推式入手，但不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列。\n\n\n\n【学生理解度分析工具结果】\n回答质量：empty\n是否理解：否\n是否答完整：否\n判定来源：fallback_heuristic\n判定理由：本总览 notebook 为了稳定"

## 这份总览 notebook 现在验证了什么

- Part 1：当前知识树和 leaf cards 确实已有正式产物可读
- Part 2：错题先系统推荐，再学生确认，最后形成规范 annotation record
- Part 3：annotation record 可以直接导入 `review_state`，并驱动 bundle 与按钮反馈
- Part 4：绑定 / 诊断 / 引导 / 复习事件可以入库，再回流成 `student_memory_profile`
- 最后：`student_memory_profile` 会轻量影响 coach 策略和 review 排序

所以这份 notebook 的定位不是“单模块调参”，而是“TeachAgent 当前四部分的总集成冒烟本”。